# TranscriptFormer cell embeddings — Open Problems batch correction

This notebook is the TranscriptFormer counterpart of `batch_corr_op.ipynb`.
It embeds the same four datasets with the species-appropriate TranscriptFormer
checkpoint: `tf-sapiens` for human datasets and `tf-exemplar` for mouse datasets.
evaluates those embeddings with the same scIB batch-correction metrics.

Datasets included: `dkd`, `gtex_v9`, `hypomap`, and `mouse_pancreas_atlas`.
`immune_cell_atlas` and `tabula_sapiens` are intentionally excluded.


## Recreate the dedicated H100 environment on Jean Zay

Run these commands once from the repository root on a login node. The Python
runtime and environment are portable because H100 nodes do not expose the
`/gpfslocalsup` Python used by V100 nodes. Package downloads occur only during
this explicit setup step; notebook execution itself is fully offline.

```bash
source /etc/profile.d/proxy.sh
export WORK="${WORK:-/lustre/fswork/projects/rech/xeg/$USER}"
export SCRATCH="${SCRATCH:-/lustre/fsn1/projects/rech/xeg/$USER}"
export UV="$HOME/.local/bin/uv"
export UV_CACHE_DIR="$SCRATCH/uv-cache"
export UV_PYTHON_INSTALL_DIR="$WORK/uv-python"
export TF_ENV="$SCRATCH/venvs/transcriptformer-h100-0.6.1"

"$UV" python install 3.11
export PYTHON_311="$WORK/uv-python/cpython-3.11.13-linux-x86_64-gnu/bin/python3.11"
"$UV" venv --python "$PYTHON_311" --relocatable "$TF_ENV"
module load r/4.4.1
bash slurm/setup_op_scib_env.sh
"$UV" pip install --python "$TF_ENV/bin/python"   transcriptformer==0.6.1 torch==2.5.1 anndata==0.11.4 scanpy==1.11.2   pandas==2.2.2 numpy==2.2.6 scipy==1.15.3 scib==1.1.7 scib-metrics==0.5.10   'jax[cuda12]==0.10.2' 'rpy2==3.5.17' 'anndata2ri==2.0.1' leidenalg==0.12.0   papermill==2.7.0 ipykernel==6.31.0 triton==3.1.0

mkdir -p "$SCRATCH/scprint_data/setuptools-overlay"
"$UV" pip install --target "$SCRATCH/scprint_data/setuptools-overlay"   setuptools==69.1.1

test -d "$WORK/models/transcriptformer/tf_sapiens"
test -d "$WORK/models/transcriptformer/tf_exemplar"
"$TF_ENV/bin/python" --version
```

TranscriptFormer 0.6.1 pins PyTorch 2.5.1. Human datasets use `tf-sapiens`;
mouse datasets use the multi-species `tf-exemplar` checkpoint. The model
receives raw, unnormalised counts and Ensembl IDs; the
preparation cell validates both conditions before inference. The shared scIB
module uses chunked JAX/GPU silhouette scores with the same OpenProblems
scaling, avoiding the multi-hour CPU silhouette calculation.


## One-H100 execution on Jean Zay

The measured configuration uses exactly one H100, mixed precision, batch size
32, and Triton block-mask compilation. Detailed inference progress is written
to one log per dataset under `$SCRATCH`, keeping every notebook result visible.

```bash
mkdir -p "$SCRATCH/scprint_data/slurm_logs"
sbatch   --job-name=tf-op-batch   --output="$SCRATCH/scprint_data/slurm_logs/tf-op-batch-%j.out"   --ntasks-per-node=1   --gres=gpu:1   --constraint=h100   --time=08:00:00   --account=wbg@h100   --nodes=1   --partition=gpu_p6   --hint=nomultithread   --qos=qos_gpu_h100-t3   --cpus-per-task=24   slurm/any_sub.sh   "slurm/run_op_scib_notebook.sh transcriptformer notebooks/scPRINT-2-repro-notebooks/batch_corr_op_transcriptformer.ipynb"
```

On a 1,024-cell benchmark, this setup completed in 37.3 seconds versus 240.9
seconds for V100 batch size 1. All data and checkpoint paths remain local.
Before running the notebook, download the four official common datasets on
the submit node with `bash slurm/download_op_common.sh`, then reconstruct
each expression reference with `slurm/run_op_no_integration.sh ...
--reference-only` inside a Slurm allocation. The notebook runner exports
`OP_SOLUTION_ROOT=$SCRATCH/openproblems_reconstructed`; PCR and cell-cycle
conservation are recomputed from these local references, not copied from an
OpenProblems solution file.


In [ ]:
from pathlib import Path
import gc
import json
import os
import shutil
import subprocess

os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "WANDB_MODE": "offline",
    "WANDB_DISABLED": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
})

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from IPython.display import display
from op_scib import (
    compute_op_scib_metrics,
    load_op_solution,
    prepare_op_scib_environment,
    save_op_scib_result,
)


In [ ]:
# Fail early if Papermill is using the wrong environment or no GPU was allocated.
subprocess.run(["transcriptformer", "--help"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["nvidia-smi"], check=True)

import importlib.metadata as metadata

print("TranscriptFormer:", metadata.version("transcriptformer"))
print("Python executable:", shutil.which("python"))
print("OpenProblems scIB environment:", prepare_op_scib_environment())


In [ ]:
WORK = Path(os.environ.get("WORK", Path.cwd()))
SCRATCH = Path(os.environ.get("SCRATCH", WORK))

DATA_ROOT = Path("data/temp")
TF_CACHE_ROOT = SCRATCH / "scprint_data"
PREPARED_ROOT = TF_CACHE_ROOT / "transcriptformer_inputs"
OUTPUT_ROOT = TF_CACHE_ROOT / "transcriptformer_outputs"
LOG_ROOT = TF_CACHE_ROOT / "transcriptformer_logs"
PCA_ROOT = TF_CACHE_ROOT / "transcriptformer_pca"
RESULT_ROOT = Path("data/results/transcriptformer")
CHECKPOINT_ROOT = WORK / "models/transcriptformer"
CHECKPOINT_BY_DATASET = {
    "cellxgene_census/dkd": CHECKPOINT_ROOT / "tf_sapiens",
    "cellxgene_census/gtex_v9": CHECKPOINT_ROOT / "tf_sapiens",
    "cellxgene_census/hypomap": CHECKPOINT_ROOT / "tf_exemplar",
    "cellxgene_census/mouse_pancreas_atlas": CHECKPOINT_ROOT / "tf_exemplar",
}

for directory in (PREPARED_ROOT, OUTPUT_ROOT, LOG_ROOT, PCA_ROOT, RESULT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

TF_BATCH_SIZE = 32
TF_INFERENCE_EXTRA_ARGS = []

datasets = (
    "cellxgene_census/dkd",
    "cellxgene_census/gtex_v9",
    "cellxgene_census/hypomap",
    "cellxgene_census/mouse_pancreas_atlas",
)

assert "cellxgene_census/immune_cell_atlas" not in datasets
assert "cellxgene_census/tabula_sapiens" not in datasets


## Prepare raw-count AnnData inputs

The official TranscriptFormer loader reads `.raw.X` first when it exists. To
make the input unambiguous, this notebook writes a temporary AnnData whose `X`
is the selected raw-count matrix and whose `var["ensembl_id"]` contains Ensembl
IDs. It also samples the matrix to reject negative or non-integer-like values.

Prepared inputs and model outputs are cached, so a restarted Slurm job resumes
at the first unfinished dataset instead of recomputing completed inference.


In [ ]:
def _sample_values(matrix, n_rows=128):
    sample = matrix[: min(n_rows, matrix.shape[0])]
    values = sample.data if sp.issparse(sample) else np.asarray(sample).ravel()
    return np.asarray(values)


def _add_ensembl_ids(adata):
    if "ensembl_id" in adata.var:
        ids = adata.var["ensembl_id"].astype(str)
    elif "ensembl_gene_id" in adata.var:
        ids = adata.var["ensembl_gene_id"].astype(str)
    elif "feature_id" in adata.var:
        ids = adata.var["feature_id"].astype(str)
    elif pd.Index(adata.var_names.astype(str)).str.startswith("ENS").all():
        ids = pd.Series(adata.var_names.astype(str), index=adata.var_names)
    else:
        raise ValueError(
            "No Ensembl IDs found: expected var['ensembl_id'], "
            "var['ensembl_gene_id'], var['feature_id'], or "
            "Ensembl-formatted var_names."
        )
    adata.var["ensembl_id"] = ids.to_numpy()


def prepare_transcriptformer_input(name):
    slug = name.rsplit("/", 1)[-1]
    source_path = DATA_ROOT / f"{name}.h5ad"
    processed_path = DATA_ROOT / f"{name}_proc.h5ad"
    prepared_path = PREPARED_ROOT / f"{slug}_raw_counts.h5ad"
    if prepared_path.exists():
        return prepared_path

    if source_path.exists():
        source = sc.read_h5ad(source_path)
        prepared = source.raw.to_adata() if source.raw is not None else source
    elif processed_path.exists():
        print(f"Using cached processed counts from {processed_path}", flush=True)
        source = sc.read_h5ad(processed_path)
        prepared = source
    else:
        raise FileNotFoundError(
            f"{name}: expected {source_path} or {processed_path}; "
            "network downloads are disabled in this notebook."
        )
    prepared.obs = source.obs.copy()
    obs_name_key = "_transcriptformer_input_obs_name"
    if obs_name_key in prepared.obs:
        raise ValueError(f"{name}: reserved obs column already exists: {obs_name_key}")
    prepared.obs[obs_name_key] = prepared.obs_names.astype(str)
    _add_ensembl_ids(prepared)

    values = _sample_values(prepared.X)
    if values.size and (
        np.nanmin(values) < 0
        or not np.allclose(values, np.rint(values), rtol=0, atol=1e-6)
    ):
        raise ValueError(
            f"{name}: TranscriptFormer requires raw, non-negative integer counts."
        )

    # Avoid a second, potentially normalized matrix taking precedence at inference.
    prepared.raw = None
    prepared.write_h5ad(prepared_path, compression="lzf")
    del source, prepared
    gc.collect()
    return prepared_path


In [ ]:
for checkpoint_path in set(CHECKPOINT_BY_DATASET.values()):
    if not checkpoint_path.is_dir():
        raise FileNotFoundError(
            f"Missing checkpoint directory: {checkpoint_path}. "
            "Network downloads are disabled in this notebook."
        )

print("Checkpoints:", CHECKPOINT_BY_DATASET)


## Run TranscriptFormer and the scIB benchmark

Each dataset runs in its own code cell and immediately displays its score table.
Verbose TranscriptFormer progress goes to `LOG_ROOT/<dataset>_inference.log`
instead of consuming the notebook output limit. Cell embeddings are read from
`obsm["embeddings"]`; scIB uses `donor_id` as batch and `cell_type` as label.


In [ ]:
def run_transcriptformer_inference(
    prepared_path, output_path, log_path, checkpoint_path
):
    """Run one-GPU inference while keeping verbose progress out of the notebook."""
    command = [
        "transcriptformer",
        "inference",
        "--checkpoint-path",
        str(checkpoint_path),
        "--data-file",
        str(prepared_path),
        "--gene-col-name",
        "ensembl_id",
        "--use-raw",
        "False",
        "--output-path",
        str(OUTPUT_ROOT),
        "--output-filename",
        output_path.name,
        "--emb-type",
        "cell",
        "--device",
        "cuda",
        "--num-gpus",
        "1",
        "--precision",
        "16-mixed",
        "--batch-size",
        str(TF_BATCH_SIZE),
        "--oom-dataloader",
        "--n-data-workers",
        "2",
        *TF_INFERENCE_EXTRA_ARGS,
    ]
    print(f"Inference log: {log_path}", flush=True)
    try:
        with log_path.open("w") as log_file:
            subprocess.run(
                command,
                check=True,
                cwd=LOG_ROOT,
                stdout=log_file,
                stderr=subprocess.STDOUT,
            )
    except subprocess.CalledProcessError:
        tail = log_path.read_text(errors="replace").splitlines()[-40:]
        print("\n".join(tail), flush=True)
        raise


def backed_row_sums(adata, chunk_size=8192):
    """Sum a backed count matrix by row without materializing it."""
    totals = np.empty(adata.n_obs, dtype=np.float64)
    for chunk, start, end in adata.chunked_X(chunk_size):
        totals[start:end] = np.asarray(chunk.sum(axis=1)).ravel()
    return totals


def benchmark_dataset(name):
    """Infer and score one cached OpenProblems dataset, returning its scIB table."""
    slug = name.rsplit("/", 1)[-1]
    checkpoint_path = CHECKPOINT_BY_DATASET[name]
    checkpoint_name = checkpoint_path.name
    output_path = OUTPUT_ROOT / f"{slug}_{checkpoint_name}_embeddings.h5ad"
    log_path = LOG_ROOT / f"{slug}_inference.log"
    score_path = RESULT_ROOT / f"{slug}_op_scib.csv"
    print(f"Dataset: {name}", flush=True)

    solution = load_op_solution(name)
    if score_path.exists():
        cached = pd.read_csv(score_path, index_col=0)
        expression_metrics = ["pcr", "cell_cycle_conservation"]
        expression_complete = cached[expression_metrics].notna().all().all()
        if solution is None or expression_complete:
            print(f"Reusing completed scores: {score_path}", flush=True)
            return cached
        print(
            f"Recomputing {score_path}: reconstructed expression reference is now available",
            flush=True,
        )

    prepared_path = prepare_transcriptformer_input(name)
    if not output_path.exists():
        run_transcriptformer_inference(
            prepared_path, output_path, log_path, checkpoint_path
        )
    else:
        print(f"Reusing completed embeddings: {output_path}", flush=True)

    embedded = ad.read_h5ad(output_path)
    prepared_backed = ad.read_h5ad(prepared_path, backed="r")
    prepared_obs_names = prepared_backed.obs_names.astype(str).to_numpy(copy=True)
    prepared_fingerprints = {
        key: prepared_backed.obs[key].to_numpy(dtype=float, copy=True)
        for key in ("nCount_RNA", "total_counts")
        if key in prepared_backed.obs
    }
    if not prepared_fingerprints:
        prepared_fingerprints["total_counts"] = backed_row_sums(
            prepared_backed
        )
    prepared_backed.file.close()
    obs_name_key = "_transcriptformer_input_obs_name"
    if obs_name_key not in embedded.obs:
        raise KeyError(f"{name}: output obs is missing {obs_name_key!r}")
    embedded_obs_names = embedded.obs.pop(obs_name_key).astype(str).to_numpy()
    if not np.array_equal(embedded_obs_names, prepared_obs_names):
        raise RuntimeError(
            f"{name}: TranscriptFormer output cell names/order differ from input; "
            "refusing to align embeddings positionally."
        )
    embedded.obs_names = embedded_obs_names
    for key, values in prepared_fingerprints.items():
        embedded.obs[key] = values
    if "embeddings" not in embedded.obsm:
        raise KeyError(f"{name}: output has no obsm['embeddings'] field")
    for required_obs in ("donor_id", "cell_type"):
        if required_obs not in embedded.obs:
            raise KeyError(f"{name}: output obs is missing {required_obs!r}")

    embedding = np.asarray(embedded.obsm.pop("embeddings"), dtype=np.float32)
    embedded.obsm["transcriptformer_emb"] = embedding
    result = compute_op_scib_metrics(
        embedded,
        embedding_key="transcriptformer_emb",
        batch_key="donor_id",
        label_key="cell_type",
        solution=solution,
        method_id="transcriptformer",
    )
    save_op_scib_result(result, score_path)
    print(f"Scores written: {score_path}", flush=True)

    del embedded, solution
    gc.collect()
    return result


### DKD


In [ ]:
dkd_result = benchmark_dataset("cellxgene_census/dkd")
display(dkd_result)


### GTEx v9


In [ ]:
gtex_v9_result = benchmark_dataset("cellxgene_census/gtex_v9")
display(gtex_v9_result)


### HypoMap


In [ ]:
hypomap_result = benchmark_dataset("cellxgene_census/hypomap")
display(hypomap_result)


### Mouse pancreas atlas


In [ ]:
mouse_pancreas_atlas_result = benchmark_dataset("cellxgene_census/mouse_pancreas_atlas")
display(mouse_pancreas_atlas_result)


In [ ]:
metrics = {
    "cellxgene_census/dkd": dkd_result,
    "cellxgene_census/gtex_v9": gtex_v9_result,
    "cellxgene_census/hypomap": hypomap_result,
    "cellxgene_census/mouse_pancreas_atlas": mouse_pancreas_atlas_result,
}
combined = pd.concat(metrics, names=["dataset", "method"])
combined_path = RESULT_ROOT / "transcriptformer_op_scib_all_datasets.csv"
combined.to_csv(combined_path)
print(f"Combined scores written: {combined_path}")
display(combined)
